## AWS External Functions in Snowflake
**Snowflake Advanced 3-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Create and test an AWS Lambda function
2. Connect an IAM-protected API Gateway REST API
3. Create a Snowflake API integration and establish AWS trust
4. Call remote Python logic from SQL, including a batch and NULL input
5. Verify against the course orders and clean up both platforms

**Run one cell at a time. Do not use Run All.** AWS console actions between cells are required. Execute Python cells in a Snowflake-hosted notebook with `snowflake-snowpark-python` available. The notebook generates policies and SQL; it does not use AWS credentials or deploy AWS resources automatically.

The request path is **SQL → external function / API integration → API Gateway → Lambda → SQL result**. This is an external function, not a Python UDF using an external access integration. The risk rule is deliberately simple; the lesson is the cross-platform connection, not predictive risk analysis.

**Prerequisites:** AWS permissions for Lambda, API Gateway REST APIs, IAM roles/policies and CloudWatch logs; Snowflake ACCOUNTADMIN for integration setup; existing course role `ADV_LAB`, warehouse `ADV_WH`, and database `ADV_ORDERS`. Use dedicated training resources. AWS services and Snowflake compute can incur charges.


### DEMO 0 | Reset a previous run (skip on first use)
Use this section only if you already created this demo. It removes the previous demo so the same final names can be used from start to finish. Do not delete course database objects beyond the specific function and integration listed below.

**Snowflake:** run the following in a SQL worksheet as ACCOUNTADMIN before starting Demo 1:

```sql
USE ROLE ACCOUNTADMIN;
DROP FUNCTION IF EXISTS ADV_ORDERS.OPS.DEMO_ORDER_RISK(FLOAT);
DROP INTEGRATION IF EXISTS ORDER_RISK_API_INTEGRATION;
```

**AWS console**, in the region used for the previous run (use the region selected for your demo):
1. API Gateway: find `snowflake-order-risk-api`, verify its identity, and delete the REST API and its stages. Confirm the API ID belongs to this demo before deleting it.
2. Lambda: open `snowflake-order-risk-demo`. Record its execution role under **Configuration → Permissions**, then use **Actions → Delete** to remove the function.
3. IAM: delete `SnowflakeOrderRiskInvokeRole` and its inline invocation policy. Delete the recorded Lambda execution role only if it was created solely for this demo and is not shared. Do not delete AWS-managed policies; remove a dedicated customer-managed logging policy only if it is unused.
4. CloudWatch Logs: delete `/aws/lambda/snowflake-order-risk-demo` if the previous logs are no longer needed.

**CHECKPOINT:** the demo API, Lambda, dedicated IAM roles, Snowflake function and API integration have been removed. Now start Demo 1 using the final names. The newly created API URL and Snowflake-generated external ID must be obtained again during this run.

This notebook does not automatically execute this reset. For cleanup after a completed run, use the final cleanup section, which uses that run's configured names.


### DEMO 1 | Set the lab names
> **[INSTRUCTOR NOTE]**
> Use the final lab names below on every run. If these resources already exist from a previous attempt, complete DEMO 0 first. Do not create a parallel set with different names.
>
> Enter your AWS account ID as a **string** so leading zeros are preserved. Set the region to the one selected in the AWS console. Neither is a credential. No secret access keys are needed.


In [ ]:
import json
import re
from urllib.parse import urlparse
from snowflake.snowpark.context import get_active_session

session = get_active_session()

aws_account_id = "024855656562"  # Replace for another AWS account; preserve leading zero.
aws_region = "us-east-2"
lambda_name = "snowflake-order-risk-demo"
api_name = "snowflake-order-risk-api"
aws_role_name = "SnowflakeOrderRiskInvokeRole"
inline_policy_name = "InvokeOrderRiskDemo"
stage_name = "demo"
integration_name = "ORDER_RISK_API_INTEGRATION"
function_name = "ADV_ORDERS.OPS.DEMO_ORDER_RISK"

assert re.fullmatch(r"\d{12}", aws_account_id)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*", integration_name)
assert re.fullmatch(r"[A-Z][A-Z0-9_]*(\.[A-Z][A-Z0-9_]*){2}", function_name)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE WAREHOUSE ADV_WH").collect()
session.sql("USE DATABASE ADV_ORDERS").collect()
session.sql("USE SCHEMA OPS").collect()
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()
print("Lambda:", lambda_name)
print("API:", api_name)
print("IAM role:", aws_role_name)
print("Snowflake function:", function_name)
session.sql("SELECT CURRENT_ROLE(), CURRENT_REGION(), CURRENT_WAREHOUSE()").show()


### DEMO 2 | Create the Lambda function in AWS
In the AWS console, select the configured region and open **Lambda**.

1. On the welcome page, click **Create an Event Function**. The “How it works” editor is only an introductory example.
2. Choose **Author from scratch**.
3. Enter the Lambda name printed above; select **Python 3.12**.
4. Leave **Durable execution** and **EC2 capacity provider** off. Leave **Additional settings** at their defaults.
5. The Permissions text says Lambda automatically creates an execution role with CloudWatch Logs permissions. Keep that default.
6. Click **Create Event Function**; dismiss the **Getting started** popup.
7. Open **Code → lambda_function.py**. Replace the starter code with the text printed by the next cell, then click **Deploy** in the editor's left panel.

This execution role lets Lambda write logs. A separate IAM role will later authorize Snowflake to invoke API Gateway.


In [ ]:
lambda_source = '''import json

def lambda_handler(event, context):
    rows = json.loads(event["body"])["data"]
    results = []

    for row_number, amount in rows:
        if amount is None:
            risk = "UNKNOWN"
        elif amount > 100:
            risk = "REVIEW"
        else:
            risk = "OK"
        results.append([row_number, risk])

    return {
        "statusCode": 200,
        "headers": {"Content-Type": "application/json"},
        "body": json.dumps({"data": results})
    }
'''
print(lambda_source)


### DEMO 3 | Test Lambda before connecting the gateway
1. Under **TEST EVENTS [NONE SELECTED]**, click **Create new test event**.
2. Name it `snowflake_batch_test`; choose **Private**.
3. Replace Event JSON with the output of the next cell.
4. Save the event. If the save action is not visible, scroll **inside the right-hand test-event form**, outside the JSON editor; the form has its own scrollbar.
5. Click the blue **Test** button below Deploy.

**CHECKPOINT:** statusCode `200`; response body contains `[[0,"REVIEW"],[1,"OK"],[2,"UNKNOWN"]]` under `data`. The body is a JSON-encoded string, so escaped quotes are normal.

Snowflake sends batches even though SQL expresses one function result per row. The first item in each input/output row is a **batch row number**, not an order ID. Return one result per input, preserving those numbers and order. Three inputs simply exercise the three branches; batch size is not fixed at three. Keep the handler free of side effects because remote requests can be retried.


In [ ]:
test_batch = {"data": [[0, 120], [1, 45], [2, None]]}
print(json.dumps({"body": json.dumps(test_batch)}, indent=2))


### DEMO 4 | Create the API Gateway REST API
1. Search for **API Gateway** in the AWS console. Keep the same region as Lambda.
2. Choose **Create API → REST API → Build** (not HTTP API or REST API Private).
3. Select **New API**, enter the configured API name, and leave Description optional.
4. Set **Endpoint type = Regional** and **IP address type = IPv4**.
5. Select **SecurityPolicy_TLS13_1_2_2021_06**, supporting TLS 1.2 and TLS 1.3.
6. Click **Create API**.

The TLS policy controls transport encryption; IAM authorization is configured separately below.


### DEMO 5 | Add the root POST method
On the Resources page, keep the root `/` selected and click **Create method**.

| Setting | Value |
|---|---|
| Method type | POST |
| Integration type | Lambda function |
| Lambda proxy integration | Enabled |
| Response transfer mode | Buffered |
| Lambda region | The configured region |
| Lambda function | The function created in Demo 2 |
| Integration timeout | 29000 ms (default) |

Expand **Method request settings**:
- Authorization: **AWS IAM**
- Request validator: **None**
- API key required: **unchecked**
- Operation name: leave blank (the displayed `GetPets` text is a placeholder)

Leave URL query strings, HTTP headers and request-body configuration unchanged. Click **Create method**. The console grants API Gateway permission to invoke Lambda through Lambda's resource-based policy.

**CHECKPOINT:** the POST Method execution page shows **Authorization = AWS_IAM**. Lambda proxy integration passes the HTTP body to our handler as `event["body"]`.


### DEMO 6 | Test API Gateway, then deploy
1. On the POST method page, select **Test**, to the right of Method response.
2. Paste the next cell's output into **Request body**; leave other optional fields empty.
3. Click **Test**.

Unlike the Lambda test, this is the raw HTTP body: do not add the outer `body` wrapper. API Gateway supplies that wrapper.

**CHECKPOINT:** HTTP `200` and the same REVIEW / OK / UNKNOWN results. This proves gateway-to-Lambda connectivity, not the Snowflake IAM trust relationship.

4. Click **Deploy API** at the upper right.
5. Choose **New stage**, name it `demo`, and click **Deploy**.
6. Copy the stage's **Invoke URL** into the cell in Demo 7. We use the root resource, so no additional resource path is appended.

An API Gateway stage is a deployment environment; it is unrelated to a Snowflake file stage. Changes to methods or API policies may require redeployment to this stage.


In [ ]:
print(json.dumps(test_batch, indent=2))

### DEMO 7 | Record the endpoint and generate the invocation policy
> **[INSTRUCTOR NOTE]**
> Use the Invoke URL of the API created during this run. A recreated API receives a new URL; do not reuse the previous endpoint. The permission is restricted to this API, stage, POST method, and root resource.


In [ ]:
invoke_url = "PASTE_INVOKE_URL_HERE"
parsed = urlparse(invoke_url)
assert parsed.scheme == "https", "Paste the deployed API Invoke URL."
assert parsed.hostname and parsed.hostname.endswith(f".execute-api.{aws_region}.amazonaws.com")
assert parsed.path.rstrip("/") == f"/{stage_name}"
assert not parsed.query and not parsed.fragment
invoke_url = invoke_url.rstrip("/")
api_id = parsed.hostname.split(".")[0]
aws_role_arn = f"arn:aws:iam::{aws_account_id}:role/{aws_role_name}"
invoke_arn = f"arn:aws:execute-api:{aws_region}:{aws_account_id}:{api_id}/{stage_name}/POST/"
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": "execute-api:Invoke", "Resource": invoke_arn}]
}
print("IAM role:", aws_role_name)
print("Policy name:", inline_policy_name)
print(json.dumps(invoke_policy, indent=2))


### DEMO 8 | Create the IAM invocation role
1. Open **IAM → Roles → Create role**.
2. Choose **AWS account → This account** as the initial trusted entity. This temporary trust is replaced in Demo 10 with the Snowflake identity.
3. Click **Next**. Select no managed permission policies; click **Next** again.
4. Enter the role name from Demo 1 and click **Create role**.
5. Open the role. Under **Permissions**, select **Add permissions → Create inline policy → JSON**.
6. Replace the policy with the JSON printed in Demo 7.
7. Click **Next**, use the printed policy name, and create the policy.
8. Verify the role ARN in the console matches the value printed below.

The policy answers **what the role may do**. The trust policy in Demo 10 answers **who may assume it**. Do not modify the separate Lambda execution role.


In [ ]:
print(aws_role_arn)

### DEMO 9 | Create the Snowflake API integration
The integration references the IAM role and limits the permitted URL prefix. Run this cell once per fresh run. It intentionally uses **CREATE**, not CREATE OR REPLACE: replacing an integration can change the external ID and require updating AWS trust again.

If the object already exists, use the following DESCRIBE cell to inspect it, or finish cleanup before starting a fresh run.


In [ ]:
session.sql("USE ROLE ACCOUNTADMIN").collect()
integration_ddl = f"""CREATE API INTEGRATION {integration_name}
    API_PROVIDER = aws_api_gateway
    API_AWS_ROLE_ARN = {sql_literal(aws_role_arn)}
    API_ALLOWED_PREFIXES = ({sql_literal(invoke_url)})
    ENABLED = TRUE"""
print(integration_ddl)
session.sql(integration_ddl).collect()
print("API integration created.")


In [ ]:
properties = {
    row["property"]: row["property_value"]
    for row in session.sql(f"DESCRIBE INTEGRATION {integration_name}").collect()
}
sf_iam_user_arn = properties["API_AWS_IAM_USER_ARN"]
external_id = properties["API_AWS_EXTERNAL_ID"]
print("API_AWS_IAM_USER_ARN:", sf_iam_user_arn)
print("API_AWS_EXTERNAL_ID:", external_id)
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"AWS": sf_iam_user_arn},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"sts:ExternalId": external_id}}
    }]
}
print("\nReplace the IAM role trust policy with:")
print(json.dumps(trust_policy, indent=2))


### DEMO 10 | Complete the AWS trust relationship
1. Return to **IAM → Roles → your invocation role**.
2. Open **Trust relationships → Edit trust policy**.
3. Replace the entire temporary policy with the JSON generated above.
4. Click **Update policy**. Leave the invocation permissions policy unchanged.

**CHECKPOINT:** the principal is the exact Snowflake-generated IAM user ARN, and `sts:ExternalId` matches the generated value. These identifiers are not AWS access keys. Never substitute values from another integration or an old run.

Return to Snowflake only after saving the trust policy. IAM changes may take a short time to propagate.


### DEMO 11 | Create the external function
Use the existing course role for function ownership and execution. The definition is stored in Snowflake; its Python implementation runs in AWS Lambda. `CALLED ON NULL INPUT` makes the NULL branch explicit. `VOLATILE` reflects that the remote implementation can change.


In [ ]:
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql(f"GRANT USAGE ON INTEGRATION {integration_name} TO ROLE ADV_LAB").collect()
session.sql("USE ROLE ADV_LAB").collect()
session.sql("USE SECONDARY ROLES NONE").collect()
session.sql("USE WAREHOUSE ADV_WH").collect()
function_ddl = f"""CREATE OR REPLACE EXTERNAL FUNCTION {function_name}(AMOUNT FLOAT)
RETURNS VARCHAR
CALLED ON NULL INPUT
VOLATILE
API_INTEGRATION = {integration_name}
AS {sql_literal(invoke_url)}"""
print(function_ddl)
session.sql(function_ddl).collect()
print("External function created.")


### DEMO 12 | Verify the complete connection
The next query tests above-threshold, below-threshold, boundary, and NULL values. The client row IDs in VALUES make the output easy to check; they are not the internal batch row numbers sent over HTTP.

**CHECKPOINT:** 120 → REVIEW, 45 → OK, 100 → OK, NULL → UNKNOWN. A successful Lambda-console or API-console test alone is not sufficient; this SQL test verifies Snowflake authentication and the full request path.


In [ ]:
test_sql = f"""SELECT COLUMN1 AS TEST_ID, COLUMN2 AS AMOUNT,
    {function_name}(COLUMN2::FLOAT) AS RISK_LABEL
FROM VALUES (1, 120), (2, 45), (3, 100), (4, NULL)
ORDER BY TEST_ID"""
print(test_sql)
test_results = session.sql(test_sql).collect()
for row in test_results:
    print(row.as_dict())
assert [r["RISK_LABEL"] for r in test_results] == ["REVIEW", "OK", "OK", "UNKNOWN"]
print("PASS: all four external-function results match.")


### DEMO 13 | Apply the function to course orders
This is a read-only demonstration. No pipeline table or task is modified. The assertion compares the remote output with our deliberately simple local SQL rule, so it does not depend on fixed order amounts.


In [ ]:
orders_sql = f"""SELECT ORDER_ID, AMOUNT,
    {function_name}(AMOUNT::FLOAT) AS RISK_LABEL,
    CASE WHEN AMOUNT IS NULL THEN 'UNKNOWN'
         WHEN AMOUNT > 100 THEN 'REVIEW' ELSE 'OK' END AS EXPECTED_LABEL
FROM ADV_ORDERS.SILVER.ORDERS_CURRENT
ORDER BY ORDER_ID"""
order_results = session.sql(orders_sql).collect()
assert order_results, "Run the course pipeline first; ORDERS_CURRENT is empty."
for row in order_results:
    print(row.as_dict())
assert all(r["RISK_LABEL"] == r["EXPECTED_LABEL"] for r in order_results)
print("PASS: remote classification matches the rule for every current order.")


### DEMO 14 | Inspect execution and explain the boundary
In Lambda, open **Monitor → View CloudWatch logs** and inspect the invocation entries. The default handler does not print input data. Snowflake may batch rows, so do not expect one invocation per table row.

Discussion:
- Snowflake manages SQL execution and authenticates through the API integration.
- API Gateway checks IAM authorization and forwards the request.
- Lambda runs the Python rule and returns batched results.
- Network calls introduce latency and failure modes. Use this for remote capabilities/services, not as a preferred replacement for a simple SQL CASE.

**Troubleshooting:**
- Access denied while creating the integration: check the Snowflake role.
- Cannot assume role: compare the IAM trust principal and external ID with DESCRIBE INTEGRATION; allow IAM propagation time.
- HTTP 403: check API authorization, invocation role permissions, API/stage/method ARN, and deployed URL.
- Missing Authentication Token: check stage/path and method; a browser issues GET but this API exposes POST.
- HTTP 500/502: check Lambda logs, proxy integration, and the response's string-valued `body`.
- KeyError `body` in Lambda: distinguish the wrapped Lambda test event from the raw API Gateway request body.
- After changing API methods/policies, redeploy to `demo` before testing again.


### DEMO CLEANUP | Remove only this lab run's resources
Run cleanup **after** verifying all checkpoints. The next cell removes the configured external function and API integration only; it does not remove course tables, roles, or the warehouse. It first prints the exact targets and requires an explicit switch.

Dropping Snowflake objects does **not** delete AWS resources. Complete the AWS console cleanup below as well.


In [ ]:
RUN_CLEANUP = False  # Set True only when this lab run is complete.
print("Function to drop:", function_name)
print("Integration to drop:", integration_name)
if RUN_CLEANUP:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql(f"DROP FUNCTION IF EXISTS {function_name}(FLOAT)").collect()
    session.sql(f"DROP INTEGRATION IF EXISTS {integration_name}").collect()
    print("Snowflake lab objects removed. Complete AWS cleanup below.")
else:
    print("Cleanup not executed. Set RUN_CLEANUP=True when ready.")


In [ ]:
print("AWS cleanup targets:")
print("Region:", aws_region)
print("API Gateway REST API:", api_name, "| API ID:", api_id)
print("Lambda:", lambda_name)
print("Invocation IAM role:", aws_role_name)
print("Inline policy:", inline_policy_name)
print("CloudWatch log group:", "/aws/lambda/" + lambda_name)


### AWS console cleanup and clean rerun
1. **API Gateway:** select the REST API by the printed name **and API ID**, then delete that API using its actions menu and confirm. This removes its deployment stages as well.
2. **Lambda:** before deleting, record the auto-created execution role under **Configuration → Permissions**. Delete the configured Lambda through **Actions → Delete**.
3. **IAM:** delete the dedicated invocation role and its inline policy. Delete the auto-created Lambda execution role only after confirming it belongs solely to this demonstration. Do not delete AWS-managed policies or a shared execution role. Remove a dedicated customer-managed logging policy only if it is unused and was created for this Lambda.
4. **CloudWatch → Logs → Log groups:** delete `/aws/lambda/<configured Lambda name>` if you do not need its logs. Lambda deletion does not automatically remove retained logs.
5. Confirm the API, Lambda and dedicated IAM roles are gone. If no other course work is running, use the course shutdown script to suspend course compute and background workloads; close the notebook session separately.

**To rehearse from scratch:** complete both Snowflake and AWS cleanup, restart the notebook kernel/session, restore `RUN_CLEANUP = False`, and start at Demo 1. The new API gets a new ID and URL: update Demo 7. Recreate the integration and regenerate the trust policy; do not reuse the old external ID. Execute each cell in order, pausing for the AWS console steps. Keep cleanup until the end.

### References
- [Snowflake: create an external function using the AWS console](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui)
- [Snowflake: Lambda setup and batch data format](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-ui-remote-service)
- [Snowflake: API integration](https://docs.snowflake.com/en/sql-reference/external-functions-creating-aws-common-api-integration)
- [AWS: API Gateway security policies](https://docs.aws.amazon.com/apigateway/latest/developerguide/apigateway-security-policies-list.html)
